# 7 — Experiment Tracking with MLflow

**Sensor Intelligence Platform** — analytical walkthrough (7 / 7)

Choosing a forecaster is an experiment, and experiments need a **ledger**: which configuration, on which sensor, scored what. `log_to_mlflow` records each backtest's parameters and metrics to MLflow so runs are comparable and reproducible rather than scattered across notebook outputs.

1. Define a small grid of forecaster configurations.
2. Backtest each on several channels and log every run to MLflow.
3. Pull the runs back into a frame and compare.
4. Pick a winner on the evidence.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (11, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})

NAVY, ORANGE, TEAL, RED, GREY = "#1f3a5f", "#e8893b", "#2a9d8f", "#c0392b", "#9aa0ad"

## 7.1 A local tracking store and a config grid

We point MLflow at a throwaway SQLite store so the notebook is self-contained and leaves no clutter in the repo. The grid pairs the two baselines against two tabular configurations differing only in lag depth.

In [2]:
import tempfile, pathlib

tmp = pathlib.Path(tempfile.mkdtemp(prefix='mlruns_'))
TRACKING_URI = f'sqlite:///{tmp.as_posix()}/mlflow.db'
EXPERIMENT = 'notebook-forecaster-grid'
print('tracking_uri =', TRACKING_URI)

PERIOD = 96
configs = [
    {'name': 'seasonal_naive', 'model': 'seasonal_naive'},
    {'name': 'rolling_mean_1d', 'model': 'rolling_mean', 'window': PERIOD},
    {'name': 'tabular_lag1d', 'model': 'tabular', 'n_lags': PERIOD},
    {'name': 'tabular_lag2d', 'model': 'tabular', 'n_lags': 2 * PERIOD},
]
sensors = ['temperature', 'flow_rate']
configs

tracking_uri = sqlite:///C:/Users/diogo/AppData/Local/Temp/mlruns_tzqe_86e/mlflow.db


[{'name': 'seasonal_naive', 'model': 'seasonal_naive'},
 {'name': 'rolling_mean_1d', 'model': 'rolling_mean', 'window': 96},
 {'name': 'tabular_lag1d', 'model': 'tabular', 'n_lags': 96},
 {'name': 'tabular_lag2d', 'model': 'tabular', 'n_lags': 192}]

## 7.2 Simulate, backtest, and log each run

For every `(config, sensor)` pair we hold out the final day with the platform's `backtest` helper and log the parameters and metrics as one MLflow run. `log_to_mlflow` returns the run id, confirming each was recorded.

In [3]:
from sensor_intelligence.simulation import SensorSimulator, SimulationConfig, default_fleet
from sensor_intelligence.domain import TimeSeriesWindow
from sensor_intelligence.models import (
    SeasonalNaiveForecaster, RollingMeanForecaster, TabularForecaster,
)
from sensor_intelligence.tracking import backtest, log_to_mlflow
import contextlib, io

fleet = [s for s in default_fleet() if s.sensor_id in sensors]
sim = SensorSimulator(SimulationConfig(sensors=fleet, n_steps=PERIOD * 12,
                                       step_seconds=900, seed=31)).run()

def window_for(sid):
    s = sim[sim.sensor_id == sid].sort_values('timestamp')
    return TimeSeriesWindow(sensor_id=sid, timestamps=list(s.timestamp),
                            values=[float(v) for v in s.value])

def forecaster(cfg):
    m = cfg['model']
    if m == 'seasonal_naive':
        return lambda w, h: SeasonalNaiveForecaster(period=PERIOD).forecast(w, h)
    if m == 'rolling_mean':
        return lambda w, h: RollingMeanForecaster(window_size=cfg['window']).forecast(w, h)
    return lambda w, h: TabularForecaster(n_lags=cfg['n_lags']).fit(w).forecast(h)

run_ids = []
# MLflow emits one-off DB-migration INFO logs to stderr on first use; mute them.
with contextlib.redirect_stderr(io.StringIO()):
    for cfg in configs:
        for sid in sensors:
            _, metrics = backtest(forecaster(cfg), window_for(sid), horizon=PERIOD)
            params = {'model': cfg['model'], 'config': cfg['name'], 'sensor': sid,
                      'n_lags': cfg.get('n_lags', ''), 'window': cfg.get('window', '')}
            rid = log_to_mlflow(params, metrics, experiment_name=EXPERIMENT,
                                run_name=f"{cfg['name']}::{sid}", tracking_uri=TRACKING_URI)
            run_ids.append(rid)
print(f'logged {len(run_ids)} runs to experiment {EXPERIMENT!r}')

logged 8 runs to experiment 'notebook-forecaster-grid'


## 7.3 Pull the runs back and compare

`mlflow.search_runs` returns every logged run as a tidy frame. We reshape it into a config-by-sensor MAE table and a leaderboard averaged across sensors — the comparison that would be guesswork from scrolling notebook outputs.

In [4]:
import mlflow

mlflow.set_tracking_uri(TRACKING_URI)
runs = mlflow.search_runs(experiment_names=[EXPERIMENT])
cols = {'params.config': 'config', 'params.sensor': 'sensor',
        'metrics.mae': 'mae', 'metrics.rmse': 'rmse',
        'metrics.interval_coverage': 'coverage'}
tidy = runs.rename(columns=cols)[list(cols.values())]
mae_table = tidy.pivot_table(index='config', columns='sensor', values='mae').round(3)
leaderboard = (tidy.groupby('config')[['mae', 'rmse', 'coverage']]
               .mean().round(3).sort_values('mae'))
print('MAE by config and sensor:')
print(mae_table)
leaderboard

MAE by config and sensor:
sensor           flow_rate  temperature
config                                 
rolling_mean_1d     11.890        4.409
seasonal_naive       3.281        0.773
tabular_lag1d        2.478        0.677
tabular_lag2d        2.426        0.771


,mae,rmse,coverage
config,,,
tabular_lag1d,1.577,1.932,0.755
tabular_lag2d,1.599,1.995,0.604
seasonal_naive,2.027,2.510,0.901
rolling_mean_1d,8.150,9.199,0.938


In [5]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
ord_ = leaderboard.index
colors = [ORANGE if c.startswith('tabular') else NAVY for c in ord_]
a1.barh(ord_, leaderboard['mae'], color=colors)
a1.invert_yaxis(); a1.set(title='Mean MAE across sensors (lower is better)', xlabel='MAE')
a2.bar(ord_, leaderboard['coverage'], color=colors)
a2.axhline(0.9, color=RED, ls='--', lw=1, label='nominal 90%')
a2.set(title='Interval coverage', ylabel='coverage', ylim=(0, 1.05))
a2.set_xticklabels(ord_, rotation=30, ha='right'); a2.legend(fontsize=9)
fig.tight_layout()

## 7.4 Pick a winner

The leaderboard makes the choice defensible: take the lowest mean MAE, then read its interval coverage as a caveat — the recursive multi-step intervals here run somewhat under the nominal 90%, a known tendency worth flagging rather than hiding. Because every run carries its parameters, the choice is **reproducible**: re-running the grid against the same store appends comparable runs rather than overwriting the record.

In [6]:
best = leaderboard.index[0]
row = leaderboard.loc[best]
print(f'Best config: {best}')
print(f'  mean MAE      : {row.mae}')
print(f'  mean RMSE     : {row.rmse}')
print(f'  mean coverage : {row.coverage}  (nominal 0.90)')

Best config: tabular_lag1d
  mean MAE      : 1.577
  mean RMSE     : 1.932
  mean coverage : 0.755  (nominal 0.90)


## Takeaways

- Every backtest is logged as an MLflow run, so model comparison is a **query**, not a scroll through cells.
- Logging **parameters with metrics** makes each result reproducible and the final choice defensible.
- A leaderboard averaged across sensors turns the grid into a single, honest ranking that respects interval calibration.
- This closes the loop opened in [notebook 2](02_forecasting_and_backtesting.ipynb): from one forecast, to a backtest, to a tracked experiment.